# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedaNehaBatool12/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token=token)

print("✅ Hugging Face login successful!")

✅ Hugging Face login successful!


In [16]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    streaming=True
)

first_row = next(iter(ds))
print(first_row)

{'client_hash_id': 'client_04660893ae39614a', 'is_active': True, 'has_gsc_access': True, 'has_ga4_access': True, 'access_profile': 'gsc_and_ga4', 'client_created_date': datetime.date(2026, 4, 15), 'client_updated_date': datetime.date(2026, 6, 27), 'gsc_data_start': None, 'ga4_data_start': datetime.date(2026, 5, 22)}


In [17]:
!pip install -q duckdb datasets huggingface_hub pyarrow

### Unit of Analysis

One row represents one pseudonymized client from the `dim_clients` table.

### Time Window

This table stores client profile and access metadata. It is not a daily performance table and is used as reference information for the warehouse.

In [18]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

print("Total clients:", len(ds))
print("Columns:")
print(ds.column_names)

Total clients: 104
Columns:
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


## Feature
- has_gsc_access
- has_ga4_access
- access_profile
- client_created_date

## Label
- is_active

## Context
- client_hash_id
- client_updated_date
- gsc_data_start
- ga4_data_start

## Excluded
None for this exploratory data contract. All available fields are documented, but only relevant fields would be used for modeling.

In [19]:
print("Columns in dim_clients:\n")

for col in ds.column_names:
    print("-", col)


Columns in dim_clients:

- client_hash_id
- is_active
- has_gsc_access
- has_ga4_access
- access_profile
- client_created_date
- client_updated_date
- gsc_data_start
- ga4_data_start


### Verification Queries

The following queries verify the data contract by checking:

1. Number of rows (grain)
2. Distribution of active and inactive clients
3. Missing values in important fields

In [20]:
from collections import Counter

# Total rows
print("Total rows:", len(ds))

# Active vs inactive clients
active_counts = Counter(ds["is_active"])
print("\nActive status:")
print(active_counts)

# Missing values
important_cols = [
    "client_hash_id",
    "access_profile",
    "gsc_data_start",
    "ga4_data_start"
]

print("\nMissing values:")

for col in important_cols:
    missing = sum(x is None for x in ds[col])
    print(f"{col}: {missing}")


Total rows: 104

Active status:
Counter({True: 74, False: 20, None: 10})

Missing values:
client_hash_id: 0
access_profile: 0
gsc_data_start: 37
ga4_data_start: 53


## Data Limits

- The dataset is pseudonymized, so client identities cannot be recovered.
- This table contains client metadata rather than daily SEO performance.
- Historical behavior cannot be inferred from this table alone.
- Some fields may have missing values depending on client access (GSC or GA4).
- Additional performance tables are required for predictive modeling.

In [21]:
important_cols = [
    "client_hash_id",
    "is_active",
    "has_gsc_access",
    "has_ga4_access",
    "access_profile"
]

print("Missing values\n")

for col in important_cols:
    missing = sum(value is None for value in ds[col])
    print(f"{col}: {missing}")

Missing values

client_hash_id: 0
is_active: 10
has_gsc_access: 10
has_ga4_access: 10
access_profile: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.